**학습 목표**: 라이브러리 없이 엔트로피와 정보이득을 계산할 수 있고, 그 값으로 "어떤 분할이 더 나은지" 판단할 수 있다.

In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer

In [50]:
def entropy(y: np.ndarray) -> float:
    """
    요구사항:
    - y는 클래스 레이블 배열(예: 0과 1).
    - 각 클래스의 비율 p_i를 구해 -sum(p_i * log2(p_i))를 계산.
    - 클래스가 한 종류만 있으면(순수 노드) 엔트로피가 0이 되어야 함.
    - p_i가 0인 경우 log2(0)이 되지 않도록 처리할 것.
    """
    values, counts = np.unique(y, return_counts = True)
    cal_counts = sum(counts)
    entropy = 0
    
    for i in range(len(counts)):
        
        if counts[i] == 0:
            entropy += 0
        else:
            entropy += -(counts[i]/cal_counts)*np.log2(counts[i]/cal_counts)

    return entropy

def information_gain(y_parent: np.ndarray, y_left: np.ndarray, y_right: np.ndarray) -> float:
    """
    요구사항:
    - 부모 노드의 엔트로피에서, 왼쪽/오른쪽 자식 노드 엔트로피의
      (샘플 수 비례) 가중평균을 뺀 값을 반환.
    """
    entropy_parent = entropy(y_parent)
    entropy_left = entropy(y_left)
    entropy_right = entropy(y_right)
    p_left = len(y_left)/len(y_parent)
    p_right = len(y_right)/len(y_parent)
    ig = entropy_parent - p_left*entropy_left - p_right*entropy_right
    return ig

data = load_breast_cancer()
y = data.target

feature_idx = data.feature_names.tolist().index("mean concave points")
feature_values = np.unique(feature_idx)

x = data.data[:,feature_idx]
y_left   = y[x >= 0.01]           
y_right  = y[x < 0.01]           


print(information_gain(y, y_left, y_right)) 

0.06267847192436105


In [27]:
import pandas as pd

In [34]:
dt = pd.DataFrame(data.data, columns=data.feature_names)
dt

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


**학습 목표**: sklearn 내부에서 일어나는 "분할점 탐색"을 직접 구현하고, 그 결과가 sklearn의 깊이 1짜리 트리(decision stump)와 같은 분할을 찾는지 검증할 수 있다.

In [97]:
def best_split(X: np.ndarray, y: np.ndarray) -> tuple[int, float, float]:
    """
    요구사항:
    - 모든 특성(열)에 대해, 정렬된 유니크값들의 중간점을 임계값 후보로 생성.
    - 각 (특성 인덱스, 임계값) 조합에 대해 Day2의 information_gain을 계산.
    - 정보이득이 최대인 (특성 인덱스, 임계값, 그때의 정보이득)을 반환.
    - 힌트: 이중 반복문으로 시작해도 되고, 익숙하면 벡터화해도 됨.
    """
    #중간값 찾기 함수
    def find_middle_point(features: np.ndarray) -> np.ndarray:
        return (features[:-1] + features[1:]) / 2

    #모든 특성(열)에 대해, 정렬된 유니크값들의 중간점을 임계값 후보로 생성
    #유니크값 생성
    uniques = []
    for i in range(X.shape[1]):
        uniques.append(np.unique(X[:,i]))
    #중간점 모음
    middle_points = []
    for i in range(len(uniques)):
        middle_points.append(find_middle_point(uniques[i]))
    #각 (특성 인덱스, 임계값) 조합에 대해 Day2의 information_gain을 계산.
    thresholds = []
    igs = []
    for i in range(X.shape[1]):
        threshold = []
        ig = []
        for j in range(len(middle_points[i])):
            x = X[:,i]
            y_left = y[x >= middle_points[i][j]]
            y_right = y[x < middle_points[i][j]]
            threshold.append(middle_points[i][j])
            ig.append(information_gain(y, y_left, y_right))
        
        
        idx = np.argmax(ig)
        thresholds.append(threshold[idx])
        igs.append(ig[idx])
    idx = np.argmax(igs)
    return tuple[idx, thresholds[idx], igs[idx]]


best_split(data.data, data.target)

tuple[np.int64(22), np.float64(105.95), np.float64(0.561986885126551)]

In [92]:
from sklearn.tree import DecisionTreeClassifier

In [95]:
tree = DecisionTreeClassifier(max_depth=1, criterion="entropy")
tree.fit(data.data, data.target)
print(tree.tree_.feature[0], tree.tree_.threshold[0])

22 105.95000076293945
